In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 02 Inspect, and turn the noisy values into NaN

In [16]:
raw = pd.read_excel("usedcars_noisy.xlsx")
raw.describe().round(1)

# -- output --
# price min: '0' cars don't sell at that price, so it's a placeholder. The mean sits below the median cause the 0s are pulling it down.
# mileage min: '-1' mileage can't be negative, so it's a sentinel value for "unknown".
# price max: plausible, roughly double on median.
# mileage max: high but plausible for a used car. It's well above the 50% mark, so it's legit high mileage rather than noise.

,year,price,mileage
count,160.0,160.0,160.0
mean,2008.8,12717.6,42031.2
std,2.2,4091.9,27954.5
min,2000.0,0.0,-1.0
25%,2008.0,10955.0,24648.8
50%,2009.0,13504.5,36120.0
75%,2010.0,14989.2,54173.5
max,2012.0,29995.0,151479.0


In [17]:
raw.columns

Index(['year', 'model', 'price', 'mileage', 'color', 'transmission'], dtype='str')

In [18]:
raw.dtypes

year            int64
model             str
price           int64
mileage         int64
color             str
transmission      str
dtype: object

### Replacing codes with NaN
All noise must be replaced with NaN before any statistics are calculated. Otherwise, the codes are averaged in as if they were real values.

In [ ]:
cars = raw.replace({"price": {0: np.nan}, "mileage": {-1: np.nan}}) # Replace 0 and -1 with NaN

# output placed in a df so it renders as a clean labelled table, with row labels (index=) and column headers. .round(1) keeps the numbers tidy too.
pd.DataFrame({
    "Codes included:" : [raw["price"].mean(), raw["mileage"].median()],
    "Codes excluded:" : [cars["price"].mean(), cars["mileage"].median()]
}, index=["Mean Price", "Median Mileage"]).round(1)

,Codes included:,Codes excluded:
Mean Price,12717.6,13127.8
Median Mileage,36120.0,36323.0


## 03 Validity and consistency
Cell checks that the data is clean after the NaN replacement, in two part. It runs on cars, the df with the noise already converted to NaN.
- first section: prints every distinct value in each text column with its count. By eye we can see there are no duplicates like 'Silver'/'silver' or 'AUTO'/'Auto' since they wouldve come up as their own key.
- second section: an assert stop shte cell with an AssertionError if the condition is false. If its true it does nothing.
    - line 1: .between(2000, 2012) gives true or false for each row, and .all() is true only if every row passes. So every year must fall within that range
    - line 2: .dropna() removes the NaNs first, because a NaN can't be compared and would make the check fail. The remaining prices must all be above 0 and mileages at least 0. This confirms that the earlier NaN replacement removed every 0 and -1

In [ ]:
for c in ["model", "color", "transmission"]:
    print(c, cars[c].value_counts().to_dict()) # value_counts() counts how many time each distinct value appears in the column, and to_dict() prints every column on a single line

assert cars["year"].between(2000,2012).all() # Ensure all years are within the expected range
assert (cars["price"].dropna() > 0).all() and (cars["mileage"].dropna() >= 0).all() # Ensure all prices are positive and mileage is non-negative
print("All categories consistent, all remaining values in range.")

model {'SE': 80, 'SES': 54, 'SEL': 26}
color {'Black': 36, 'Silver': 35, 'Red': 27, 'White': 18, 'Blue': 18, 'Gray': 17, 'Green': 5, 'Yellow': 3, 'Gold': 1}
transmission {'AUTO': 135, 'MANUAL': 25}
All categories consistent, all remaining values in range.


### Checking duplicates

In [26]:
print("duplicate rows:", cars.duplicated().sum()) # Check for duplicate rows

duplicate rows: 0


### Diagnose missing values
Eight of the ten rows with gaps are 2010 or 2011 cars, against 46% of the whole file. WIth only ten rows that is thin evidence, but it suggests the gaps are linked to year (MAR) rather than completely random.

In [27]:
gaps = cars["price"].isna() | cars["mileage"].isna() # Identify rows with missing price or mileage

print("Missing price:", cars["price"].isna().sum(), "| missing mileage:", cars["mileage"].isna().sum(), "| both:", (cars["price"].isna() & cars["mileage"].isna()).sum())
print("Share of 2010 to 2011 cars, rows with gaps:", f"{cars.loc[gaps, 'year'].between(2010, 2011).mean():.0%}", "| all rows:", f"{cars['year'].between(2010, 2011).mean():.0%}")

Missing price: 5 | missing mileage: 7 | both: 2
Share of 2010 to 2011 cars, rows with gaps: 80% | all rows: 46%


In [ ]:
cars[gaps]

# the gaps are the NaN values in the price and mileage columns. The rows with gaps are mostly 2010 and 2011 cars, which is a bit odd. It could be that the data was collected from 
# a source that didn't have complete information for those years.

,year,model,price,mileage,color,transmission
2,2011,SEL,29995.0,NaN,Gray,AUTO
7,2010,SES,NaN,NaN,Silver,AUTO
11,2011,SES,NaN,4232.0,Blue,AUTO
22,2011,SE,16901.0,NaN,White,AUTO
27,2010,SES,15228.0,NaN,Silver,MANUAL
33,2010,SEL,NaN,65636.0,Black,MANUAL
43,2010,SEL,19906.0,NaN,White,AUTO
157,2010,SES,NaN,16035.0,Red,MANUAL
158,2005,SES,8488.0,NaN,Red,AUTO
159,2004,SE,NaN,NaN,Silver,AUTO


The above matters because price and mileage both depend strongly on year:
Newer cars cost more (correlation 0.80) and havce driven less (-0.76). A single overall value ignores this: a 2011 cars and a 2004 car with a missing price both get the same mean price. A method that uses similar cars should better. The 'choosing an imputation method' step tests that.

In [28]:
cars[["year","price","mileage"]].corr().round(2) # Check correlation between year, price and mileage

,year,price,mileage
year,1.00,0.80,-0.76
price,0.80,1.00,-0.81
mileage,-0.76,-0.81,1.00


## 04 Outliers
This block finds values in price and mileage that are unusually high or low, using the IQR rule. It prints them but doesn't change or remove anything.

First section (def method): 
- q1 and q3 are the 25th and 75th percentiles
- q3 - q1 is the IQR, the spread of the middle 50% of the data
- It returns two fences:
    - lower = Q1 - 1.5 x IQR
    - upper = Q3 + 1.5 x IQR
- k=1.5 is the standard multiplier. A larger k such as 3 only flags very extreme values
Anything outside the fences counts as a potential outlier

Section second (for loop): for each column it
- calculates the fences. Because it runs on cars, the NaNs are skipped, so the 0 and 01 codes don't distort the quartiles.
- selects the rows below the fence or above
- prints the fence range, e.g. price: fences 5001 to 20985, then the rows below or above the fences are the outliers and displayed.

Output analysis: the outliers are older cars that are cheap with high mileage, and three 2011 SELs (the top trim) at the top fo the price range. They are consistent with their year and trim, so they are real and are kept


In [ ]:
def iqr_fences(s, k=1.5): # s represents a pandas Series, k is the multiplier for the IQR to determine the fences
    q1, q3 = s.quantile([0.25, 0.75]) # Calculate the first and third quartiles (25th and 75th percentiles) of the Series
    return q1 - k * (q3 - q1), q3 + k * (q3 - q1) # Return the lower and upper fences for outlier detection based on the IQR method

for c in ["price", "mileage"]:
    lower, upper = iqr_fences(cars[c]) # Calculate the lower and upper fences for each column using the IQR method
    out = cars[(cars[c] < lower) | (cars[c] > upper)] # Identify outliers based on the IQR fences
    print(f"{c}: fences {lower:,.0f} to {upper:,.0f}") # print the lower and upper fences for each column, formatted with commas and no decimal places
    print(out[["year","model", "price", "mileage"]].to_string(), "\n") # print the outliers for each column, showing year, model, price, and mileage


price: fences 5,001 to 20,985
     year model    price   mileage
0    2011   SEL  21992.0    7413.0
1    2011   SEL  20995.0   10926.0
2    2011   SEL  29995.0       NaN
155  2001    SE   4899.0  151479.0
156  2000    SE   3800.0  109259.0 

mileage: fences -15,538 to 97,594
     year model    price   mileage
96   2008    SE  12995.0  127327.0
139  2008    SE   8495.0  106171.0
144  2006   SES   7995.0   97987.0
147  2004    SE   6999.0  105714.0
150  2004   SES   6980.0  101130.0
151  2004   SES   6950.0  119720.0
155  2001    SE   4899.0  151479.0
156  2000    SE   3800.0  109259.0 

